In [ ]:
import openpyxl
from openpyxl import load_workbook
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

In [ ]:
feature_cols = [
    "PTS",  "REB",  "AST",  "STL",
    "BLK",  "TOV",  "PF",   "FG%",
    "FGA",  "2P%",  "2PA",  "3P%",
    "3PA",  "FT%",  "FTA",  "USG"
]
test_players = [
        'Shai Gileous-Alexander', 'Dejounte Murray', 'Trae Young', 'Luka Doncic', 'Fred VanVleet',
        "De'Aaron Fox", 'Jalen Brunson', 'Jaylen Brown', 'Donovan Mitchell', 'Jayson Tatum', 'Brandon Ingram',
        'Pascal Siakam', 'Lauri Markkanen', 'Bam Adebayo', 'Domantas Sabonis', 'Jarrett Allen', 'Jaren Jackson Jr.'
    ]

In [ ]:
sheets_in = ["First", "Second", "Third"]
dfs_in = [pd.read_excel("NBA_allStar.xlsx", sheet_name=s).drop(columns=["Positions"]) for s in sheets_in]
df_out  = (pd.read_excel("NBA_allStar.xlsx", sheet_name="A3").drop(columns=["Positions"], errors="ignore"))

In [ ]:
for df in dfs_in + [df_out]:
    df.sort_values("Player", inplace=True)
    df.reset_index(drop=True, inplace=True)
# --- 完成 feature_cols 與各 DataFrame 讀取、排序後 ---
mask_test = df_out['Player'].isin(test_players)

# 驗證是否全命中
missing = set(test_players) - set(df_out.loc[mask_test, 'Player'])
if missing:
    print("以下名字在資料中找不到：", missing)

X = np.hstack([df[feature_cols].values for df in dfs_in])   # (N, 48)
y = df_out[feature_cols].values       # (N, 16)

In [ ]:
# ------------- (A) 用遮罩切 Raw -------------
X_raw_train = X[~mask_test]
X_raw_test  = X[mask_test]
y_raw_train = y[~mask_test]
y_raw_test  = y[mask_test]
# ------------- (B) Fit 標準化器 -------------
scaler_X = StandardScaler().fit(X_raw_train)   # 只用訓練集
scaler_y = StandardScaler().fit(y_raw_train)

X_train = scaler_X.transform(X_raw_train)
X_test  = scaler_X.transform(X_raw_test)
y_train = scaler_y.transform(y_raw_train)
y_test  = scaler_y.transform(y_raw_test)

In [ ]:
mlp = MLPRegressor(hidden_layer_sizes=(64, 32),
                   activation="relu",
                   solver="adam",
                   max_iter=2000,
                   early_stopping=False,
                   n_iter_no_change=50,
                   random_state=42)
mlp.fit(X_train, y_train)

MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000, n_iter_no_change=50,
             random_state=42)

In [ ]:
y_pred_test = scaler_y.inverse_transform(mlp.predict(X_test))
y_test_orig = scaler_y.inverse_transform(y_test)

In [ ]:
# test_players_mask 與 y_pred_test 都已建立
player_names_test = df_out.loc[mask_test, 'Player'].values    # 依遮罩取名字

df_pred_test = pd.DataFrame(
    y_pred_test,                         # 預測值 (已 inverse_transform)
    index=player_names_test,             # 把名字放在 index
    columns=feature_cols                 # 16 個欄位名
)

df_pred_test["FGA"] = df_pred_test["2PA"] + df_pred_test["3PA"]
df_pred_test['PTS'] = 2*df_pred_test['2PA']*df_pred_test['2P%'] + 3*df_pred_test['3PA']*df_pred_test['3P%'] + df_pred_test['FTA']*df_pred_test['FT%']
print(df_pred_test.head())               # 先看前幾筆
# 若要全部印，可直接 print(df_pred_test)

# 需要輸出 Excel：
df_pred_test.to_excel("MLP_predictions.xlsx")

                        PTS        REB       AST       STL       BLK  \
Bam Adebayo       20.054006   9.393765  5.394184  1.028108  1.547070   
Brandon Ingram    16.437521   6.387840  5.458796  0.991732  0.495755   
De'Aaron Fox      22.026876   6.207733  8.440324  1.529640  0.557859   
Dejounte Murray   18.884258   7.658837  4.485574  1.883326  0.419709   
Domantas Sabonis  22.267592  11.481080  5.434961  0.676773  1.095078   

                       TOV        PF       FG%        FGA       2P%  \
Bam Adebayo       2.135686  2.583668  0.551161  14.054722  0.568273   
Brandon Ingram    3.391276  3.166143  0.498043  13.358701  0.510232   
De'Aaron Fox      3.991357  2.365599  0.445757  17.701056  0.496963   
Dejounte Murray   3.222850  3.554339  0.495859  14.777116  0.483664   
Domantas Sabonis  3.573019  4.629342  0.579897  15.980531  0.593184   

                        2PA       3P%       3PA       FT%       FTA        USG  
Bam Adebayo       13.908288  0.229277  0.146433  0.725906  

In [ ]:
df_test_true = pd.DataFrame(
    y_test_orig,                  # inverse_transform 後的真值
    index=player_names_test,
    columns=feature_cols
)

In [ ]:
epsilon = 1e-12
denom = df_test_true.replace(0, epsilon)   # 對 0 做替換

mape_df = (df_pred_test.sub(df_test_true).abs()
           .div(denom)
           .mean(axis=0) * 100)            # axis=0 → 各欄位平均

df_mape = mape_df.reset_index()
df_mape.columns = ["Feature", "MAPE_%"]
df_mape = df_mape.sort_values("MAPE_%")

print(df_mape)
print(f"\n新 MAP E 全欄平均 = {df_mape['MAPE_%'].mean():.2f} %")

   Feature     MAPE_%
9      2P%   5.198364
7      FG%   5.452182
13     FT%   6.781342
15     USG  11.312944
8      FGA  14.475702
0      PTS  15.104359
1      REB  15.981035
10     2PA  16.562924
14     FTA  18.081099
6       PF  18.344984
11     3P%  19.206066
3      STL  22.559299
2      AST  23.606849
5      TOV  25.147741
12     3PA  43.178304
4      BLK  57.396620

新 MAP E 全欄平均 = 19.90 %


In [ ]:
with pd.ExcelWriter("MLP_predictions.xlsx", mode="a",
                    engine="openpyxl", if_sheet_exists="replace") as writer:
    df_pred_test.to_excel(writer, sheet_name="Predictions_updated")
    df_mape.to_excel(writer, sheet_name="MAPE_updated", index=False)

In [ ]:
from sklearn.neural_network import MLPRegressor
import numpy as np

mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=1,           # 每次只跑 1 iter
    warm_start=True,      # 連續呼叫 partial_fit 會延續權重
    random_state=42,
    alpha=1e-1            # 視需要加強 L2 正則化
)

In [ ]:
target_mape = 10.0           # 希望 MAPE ≤ 10 %
max_epochs  = 270
patience    = 50             # 連續 50 epoch 沒改善就停
wait        = 0
best_mape   = np.inf         # 初始設很大

epsilon = 1e-12              # 真值為 0 的替代分母

for epoch in range(1, max_epochs + 1):

    mlp.partial_fit(X_train, y_train)

    # ---------- 驗證集預測 ----------
    y_pred_val = mlp.predict(X_test)
    y_pred_val_inv = scaler_y.inverse_transform(y_pred_val)
    y_val_inv      = scaler_y.inverse_transform(y_test)

    # ---------- 計算平均 MAPE ----------
    denom = np.where(np.abs(y_val_inv) < 1e-12, epsilon, y_val_inv)
    mape  = np.mean(np.abs((y_val_inv - y_pred_val_inv) / denom)) * 100

    # 每 10 epoch 印一次狀況
    if epoch % 10 == 0 or mape < target_mape:
        print(f"Epoch {epoch:4d}  Val MAPE = {mape:6.2f} %")

    # ---------- 早停邏輯 ----------
    if mape < best_mape - 1e-4:          # 有顯著改善
        best_mape, wait = mape, 0
    else:
        wait += 1

    if mape <= target_mape:
        print(f"\n✅ 目標達成！Val MAPE = {mape:.2f} % ≤ {target_mape}% —— 提前停止")
        break

    if wait >= patience:
        print(f"\n⏹️ 連續 {patience} epoch 無明顯改善，停止於 Epoch {epoch}")
        break


Epoch   10  Val MAPE =  45.22 %
Epoch   20  Val MAPE =  40.75 %
Epoch   30  Val MAPE =  35.76 %
Epoch   40  Val MAPE =  31.80 %
Epoch   50  Val MAPE =  28.90 %
Epoch   60  Val MAPE =  27.95 %
Epoch   70  Val MAPE =  25.61 %
Epoch   80  Val MAPE =  23.14 %
Epoch   90  Val MAPE =  21.82 %
Epoch  100  Val MAPE =  21.29 %
Epoch  110  Val MAPE =  20.65 %
Epoch  120  Val MAPE =  19.87 %
Epoch  130  Val MAPE =  19.37 %
Epoch  140  Val MAPE =  19.02 %
Epoch  150  Val MAPE =  18.96 %
Epoch  160  Val MAPE =  18.86 %
Epoch  170  Val MAPE =  18.72 %
Epoch  180  Val MAPE =  18.65 %
Epoch  190  Val MAPE =  18.64 %
Epoch  200  Val MAPE =  18.63 %
Epoch  210  Val MAPE =  18.54 %
Epoch  220  Val MAPE =  18.52 %
Epoch  230  Val MAPE =  18.52 %
Epoch  240  Val MAPE =  18.50 %
Epoch  250  Val MAPE =  18.50 %
Epoch  260  Val MAPE =  18.56 %
Epoch  270  Val MAPE =  18.60 %


In [ ]:
player_names_test = df_out.loc[mask_test, 'Player'].values    # 依遮罩取名字

df_pred_test = pd.DataFrame(
    y_pred_val_inv,                         # 預測值 (已 inverse_transform)
    index=player_names_test,             # 把名字放在 index
    columns=feature_cols                 # 16 個欄位名
)
#df_pred_test["2PA"] = df_pred_test["2PA"]*1.29
#df_pred_test["3PA"] = df_pred_test["3PA"]*0.83
df_pred_test["FGA"] = df_pred_test["2PA"] + df_pred_test["3PA"]
df_pred_test['PTS'] = 2*df_pred_test['2PA']*df_pred_test['2P%'] + 3*df_pred_test['3PA']*df_pred_test['3P%'] + df_pred_test['FTA']*df_pred_test['FT%']
print(df_pred_test.head())               # 先看前幾筆
# 若要全部印，可直接 print(df_pred_test)

# 需要輸出 Excel：
df_pred_test.to_excel("MLP_predictions.xlsx")

                        PTS        REB       AST       STL       BLK  \
Bam Adebayo       19.952933   9.534492  4.886118  1.007720  1.514372   
Brandon Ingram    16.946910   6.611515  5.482221  1.056381  0.534627   
De'Aaron Fox      22.381778   6.341819  8.638641  1.546818  0.621457   
Dejounte Murray   18.626280   7.264447  4.767697  1.857344  0.359343   
Domantas Sabonis  22.157253  11.382737  4.906376  0.713998  1.140611   

                       TOV        PF       FG%        FGA       2P%  \
Bam Adebayo       2.133754  2.642969  0.557415  14.010777  0.570120   
Brandon Ingram    3.382798  3.015151  0.487425  13.717136  0.508348   
De'Aaron Fox      3.997907  2.360416  0.440070  17.569722  0.505630   
Dejounte Murray   3.248423  3.507326  0.496422  14.581396  0.482237   
Domantas Sabonis  3.550424  4.515536  0.571199  16.241476  0.587524   

                        2PA       3P%       3PA       FT%       FTA        USG  
Bam Adebayo       13.885068  0.235334  0.125709  0.722711  

In [ ]:
epsilon = 1e-12
denom = df_test_true.replace(0, epsilon)   # 對 0 做替換

mape_df = (df_pred_test.sub(df_test_true).abs()
           .div(denom)
           .mean(axis=0) * 100)            # axis=0 → 各欄位平均

df_mape = mape_df.reset_index()
df_mape.columns = ["Feature", "MAPE_%"]
df_mape = df_mape.sort_values("MAPE_%")

print(df_mape)
print(f"\n新 MAPE 全欄平均 = {df_mape['MAPE_%'].mean():.2f} %")

   Feature     MAPE_%
9      2P%   4.844636
7      FG%   5.642304
13     FT%   6.385030
15     USG  10.351292
0      PTS  13.692881
8      FGA  14.019225
1      REB  14.862329
10     2PA  15.478957
14     FTA  16.066668
6       PF  17.447338
11     3P%  18.687318
2      AST  19.655707
3      STL  21.462984
5      TOV  24.652790
12     3PA  35.431351
4      BLK  62.999973

新 MAPE 全欄平均 = 18.86 %


In [ ]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train = torch.tensor(y_train, dtype=torch.float32, device=device)
X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
y_test = torch.tensor(y_test, dtype=torch.float32, device=device)


In [ ]:
import os, random

def seed_everything(seed: int = 2025):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)       # 影響 set/dict、hash
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 讓 cuDNN 使用「可重現」的 kernel，而非最快但可能隨機的版本
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False         # 關掉自動挑最快 kernel
    # PyTorch 1.12+ 亦可：
    # torch.use_deterministic_algorithms(True)

seed_everything(2025)          # ← 先呼叫，再做任何事


In [ ]:
class MLPRegressor(torch.nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.net = torch.nn.Sequential(
            *[torch.nn.Sequential(torch.nn.Linear(layers[i], layers[i+1]),
                                   torch.nn.ReLU()) #torch.nn.Tanh torch.nn.ReLU()
              for i in range(len(layers)-2)],
            torch.nn.Linear(layers[-2], layers[-1])
        )
        # Xavier 初始化
        for m in self.net:
            if isinstance(m, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight)
                torch.nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)

In [ ]:
class MAPELoss(torch.nn.Module):
    def __init__(self, eps=1e-12):
        super().__init__()
        self.eps = eps                # 防除以 0
    def forward(self, pred, target):
        return torch.mean(torch.abs((pred - target) / (target + self.eps))) * 100

model = MLPRegressor(layers=[48, 64, y.shape[1]]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-2)
criterion  = torch.nn.MSELoss() #torch.nn.MSELoss()MAPELoss()

n_epochs = 22000
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train.to(device))
    loss = criterion(pred, y_train.to(device))
    loss.backward()
    optimizer.step()

    # 每 2000 回合列印一次 & 驗證
    if epoch % 2000 == 0:
        model.eval()
        with torch.no_grad():
            val_pred = model(X_test.to(device))
            val_loss = criterion(val_pred, y_test.to(device))
        print(f"[{epoch}] train MSE = {loss.item():.4f}, test MSE = {val_loss.item():.4f}")


[0] train MSE = 1.1498, test MSE = 1.2099
[2000] train MSE = 0.5125, test MSE = 0.6544
[4000] train MSE = 0.2848, test MSE = 0.4685
[6000] train MSE = 0.1779, test MSE = 0.4011
[8000] train MSE = 0.1191, test MSE = 0.3732
[10000] train MSE = 0.0862, test MSE = 0.3579
[12000] train MSE = 0.0670, test MSE = 0.3476
[14000] train MSE = 0.0551, test MSE = 0.3446
[16000] train MSE = 0.0477, test MSE = 0.3423
[18000] train MSE = 0.0430, test MSE = 0.3437
[20000] train MSE = 0.0397, test MSE = 0.3468


In [ ]:
y_pred_test = scaler_y.inverse_transform(val_pred)
y_test_orig = scaler_y.inverse_transform(y_test)

In [ ]:
player_names_test = df_out.loc[mask_test, 'Player'].values    # 依遮罩取名字

df_pred_test = pd.DataFrame(
    y_pred_test,                         # 預測值 (已 inverse_transform)
    index=player_names_test,             # 把名字放在 index
    columns=feature_cols                 # 16 個欄位名
)
df_pred_test["FGA"] = df_pred_test["2PA"] + df_pred_test["3PA"]
df_pred_test['PTS'] = 2*df_pred_test['2PA']*df_pred_test['2P%'] + 3*df_pred_test['3PA']*df_pred_test['3P%'] + df_pred_test['FTA']*df_pred_test['FT%']
print(df_pred_test.head())               # 先看前幾筆
# 若要全部印，可直接 print(df_pred_test)

# 需要輸出 Excel：
df_pred_test.to_excel("MLP_predictions.xlsx")

                        PTS        REB       AST       STL       BLK  \
Bam Adebayo       18.471746   9.967756  4.411697  1.288178  1.352825   
Brandon Ingram    21.523136   6.709173  4.564475  0.900687  0.680882   
De'Aaron Fox      23.450423   5.821667  8.368071  1.564591  0.572389   
Dejounte Murray   17.925076   6.952336  6.950133  1.839656  0.657642   
Domantas Sabonis  25.322859  11.496667  5.669808  1.020837  1.039028   

                       TOV        PF       FG%        FGA       2P%  \
Bam Adebayo       2.437990  3.045346  0.552909  12.594131  0.564836   
Brandon Ingram    2.950680  2.867473  0.495285  16.556962  0.530975   
De'Aaron Fox      3.720193  2.479514  0.469960  17.867116  0.506093   
Dejounte Murray   2.960736  2.894426  0.461278  14.815895  0.492430   
Domantas Sabonis  3.672977  4.029549  0.533346  17.230698  0.593024   

                        2PA       3P%       3PA       FT%       FTA        USG  
Bam Adebayo       12.177466  0.197132  0.416665  0.720188  

In [ ]:
epsilon = 1e-12
denom = df_test_true.replace(0, epsilon)   # 對 0 做替換

mape_df = (df_pred_test.sub(df_test_true).abs()
           .div(denom)
           .mean(axis=0) * 100)            # axis=0 → 各欄位平均

df_mape = mape_df.reset_index()
df_mape.columns = ["Feature", "MAPE_%"]
df_mape = df_mape.sort_values("MAPE_%")

print(df_mape)
print(f"\n新 MAPE 全欄平均 = {df_mape['MAPE_%'].mean():.2f} %")   #(||X - X1||/||X||)

   Feature     MAPE_%
7      FG%   4.698383
13     FT%   5.014130
9      2P%   5.106043
15     USG  10.045911
10     2PA  10.722175
0      PTS  10.887009
8      FGA  11.330275
14     FTA  11.855355
2      AST  13.513470
6       PF  13.715806
1      REB  13.969008
11     3P%  14.005441
3      STL  14.134002
5      TOV  16.978952
12     3PA  56.905248
4      BLK  74.698385

新 MAPE 全欄平均 = 17.97 %


In [ ]:
sum = 0
for items in test_players:
  norm_vec_np = np.linalg.norm(df_test_true.T[items] - df_pred_test.T[items]) #L-2 norm
  norm_vec_true = np.linalg.norm(df_test_true.T[items])
  A = norm_vec_np/norm_vec_true
  sum += A
  print(items, ':', A)
print(sum/17)

Shai Gileous-Alexander : 0.14732690084914415
Dejounte Murray : 0.1098456529132234
Trae Young : 0.031304260720994674
Luka Doncic : 0.06117687663610868
Fred VanVleet : 0.07195899319622645
De'Aaron Fox : 0.08214972023780903
Jalen Brunson : 0.0658150152447382
Jaylen Brown : 0.12525218868842045
Donovan Mitchell : 0.08198797104875565
Jayson Tatum : 0.07927373225754465
Brandon Ingram : 0.12144056045426212
Pascal Siakam : 0.21345339792598234
Lauri Markkanen : 0.2530531922594213
Bam Adebayo : 0.10572037151329784
Domantas Sabonis : 0.23148919287815015
Jarrett Allen : 0.18456534751800932
Jaren Jackson Jr. : 0.08445216692746796
0.12060385536879743


In [ ]:
#扣除BLK和3PA，MAPE為11%

In [ ]:
MAPE_PTS = abs((df_pred_test['PTS'] - df_test_true['PTS']))/df_test_true['PTS']
print(MAPE_PTS*100)
print('---------------------------')
MAPE_REB = abs((df_pred_test['REB'] - df_test_true['REB']))/df_test_true['REB']
print(MAPE_REB*100)
print('---------------------------')
MAPE_AST = abs((df_pred_test['AST'] - df_test_true['AST']))/df_test_true['AST']
print(MAPE_AST*100)

Bam Adebayo               11.193527
Brandon Ingram            14.250453
De'Aaron Fox               8.396783
Dejounte Murray           10.374622
Domantas Sabonis          28.542430
Donovan Mitchell           0.611967
Fred VanVleet              3.105748
Jalen Brunson              1.062893
Jaren Jackson Jr.          7.676503
Jarrett Allen              7.713766
Jaylen Brown              14.689463
Jayson Tatum               7.314212
Lauri Markkanen           24.881066
Luka Doncic                1.816139
Pascal Siakam             21.703183
Shai Gileous-Alexander    18.429384
Trae Young                 3.317016
Name: PTS, dtype: float64
---------------------------
Bam Adebayo                0.322440
Brandon Ingram            15.675397
De'Aaron Fox              41.991867
Dejounte Murray            2.079773
Domantas Sabonis           8.026664
Donovan Mitchell           4.494253
Fred VanVleet              9.715871
Jalen Brunson             17.225114
Jaren Jackson Jr.         33.026813
Jarrett Al